In [ ]:
import pandas as pd

import torch
from torch.nn import CrossEntropyLoss

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoModelForCausalLM,
    pipeline,
    Trainer,
    TrainingArguments
)

from datasets import Dataset
import pandas as pd
import numpy as np
import regex as re

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score
from sklearn.utils.class_weight import compute_class_weight

Teste de gpu

In [ ]:
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(device)

True
Tesla T4
cuda


# Tratando amostra do dataset já classificada

Limpeza do Data Frame


In [ ]:
def preprocess_transformer(text):
    text = str(text)
    text = text.strip()
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
df = pd.read_csv("LINK DO ARQUIVO NO DRIVE")

if 'textClean' not in df.columns:
    df['textClean'] = None

df['commentText'] = df['commentText'].str.replace(r"http\S+|www\S+", " ", regex=True)

df['textClean'] = df['commentText'].apply(preprocess_transformer)

label_map = {"negativo": 0, "neutro": 1, "positivo": 2}
df['feeling'] = df['feeling'].map(label_map)

Divisão e Cross-Validation

In [ ]:

X = df['textClean']
y = df['feeling']

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

splits = list(skf.split(X, y))

Class Weights

In [ ]:
classes = np.unique(y)

weights = compute_class_weight(
    class_weight='balanced',
    classes=classes,
    y=y
)

class_weights = torch.tensor(weights, dtype=torch.float)

print("Class weights:", class_weights)

Class weights: tensor([1.8165, 0.4521, 4.2128])


Weighted Trainer

In [ ]:
class WeightedTrainer(Trainer):
    def compute_loss(
        self,
        model,
        inputs,
        return_outputs=False,
        num_items_in_batch=None
    ):
        labels = inputs.get("labels")

        outputs = model(**inputs)

        logits = outputs.get("logits")

        loss_fct = CrossEntropyLoss(
            weight=class_weights.to(logits.device)
        )

        loss = loss_fct(logits, labels)

        return (loss, outputs) if return_outputs else loss

# BERTimbau

In [ ]:
bert = "neuralmind/bert-base-portuguese-cased"

bert_tokenizer = AutoTokenizer.from_pretrained(bert)

config.json:   0%|          | 0.00/647 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/43.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [ ]:
f1_scores_bert = []

def tokenize_bert(batch):
    return bert_tokenizer(batch["text"], truncation=True, padding=True, max_length=128)

for train_idx, test_idx in splits:

    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    train_dataset = Dataset.from_dict({
        "text": X_train.tolist(),
        "label": y_train.tolist()
    })

    test_dataset = Dataset.from_dict({
        "text": X_test.tolist(),
        "label": y_test.tolist()
    })

    train_dataset = train_dataset.map(tokenize_bert, batched=True)
    test_dataset  = test_dataset.map(tokenize_bert, batched=True)

    train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
    test_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

    model = AutoModelForSequenceClassification.from_pretrained(bert, num_labels=3).to(device)

    training_args = TrainingArguments(
        output_dir="./results_bert",
        per_device_train_batch_size=8,
        num_train_epochs=3,
        logging_steps=50,
        save_strategy="no",
        seed=42,
        disable_tqdm=True
    )

    trainer = WeightedTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=test_dataset
    )

    trainer.train()

    preds_output = trainer.predict(test_dataset)
    preds = np.argmax(preds_output.predictions, axis=1)

    f1 = f1_score(y_test, preds, average='macro')
    f1_scores_bert.append(f1)

print("BERTimbau F1 médio:", np.mean(f1_scores_bert))
print("Desvio padrão:", np.std(f1_scores_bert))

Map:   0%|          | 0/475 [00:00<?, ? examples/s]

Map:   0%|          | 0/119 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: neuralmind/bert-base-portuguese-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. C

{'loss': '1.096', 'grad_norm': '8.148', 'learning_rate': '3.639e-05', 'epoch': '0.8333'}
{'loss': '1.038', 'grad_norm': '10.52', 'learning_rate': '2.25e-05', 'epoch': '1.667'}
{'loss': '0.8531', 'grad_norm': '8.824', 'learning_rate': '8.611e-06', 'epoch': '2.5'}
{'train_runtime': '33.02', 'train_samples_per_second': '43.15', 'train_steps_per_second': '5.451', 'train_loss': '0.9593', 'epoch': '3'}


...

Map:   0%|          | 0/476 [00:00<?, ? examples/s]

Map:   0%|          | 0/118 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: neuralmind/bert-base-portuguese-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. C

{'loss': '1.048', 'grad_norm': '9.757', 'learning_rate': '3.639e-05', 'epoch': '0.8333'}
{'loss': '0.8553', 'grad_norm': '13.27', 'learning_rate': '2.25e-05', 'epoch': '1.667'}
{'loss': '0.5105', 'grad_norm': '9.013', 'learning_rate': '8.611e-06', 'epoch': '2.5'}
{'train_runtime': '35.95', 'train_samples_per_second': '39.72', 'train_steps_per_second': '5.006', 'train_loss': '0.7302', 'epoch': '3'}
BERTimbau F1 médio: 0.5290819206928953
Desvio padrão: 0.04458834986649376


# XLM-RoBERTa

In [ ]:
roberta = "xlm-roberta-base"

roberta_tokenizer = AutoTokenizer.from_pretrained(roberta)

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
f1_scores_roberta = []

def tokenize_roberta(batch):
    return roberta_tokenizer(batch["text"], truncation=True, padding=True, max_length=128)

for train_idx, test_idx in splits:

    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    train_dataset = Dataset.from_dict({
        "text": X_train.tolist(),
        "label": y_train.tolist()
    })

    test_dataset = Dataset.from_dict({
        "text": X_test.tolist(),
        "label": y_test.tolist()
    })

    train_dataset = train_dataset.map(tokenize_roberta, batched=True)
    test_dataset  = test_dataset.map(tokenize_roberta, batched=True)

    train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
    test_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

    model = AutoModelForSequenceClassification.from_pretrained(roberta, num_labels=3).to(device)

    training_args = TrainingArguments(
        output_dir="./results_roberta",
        per_device_train_batch_size=8,
        num_train_epochs=3,
        logging_steps=50,
        save_strategy="no",
        seed=42,
        disable_tqdm=True
    )

    trainer = WeightedTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=test_dataset
    )

    trainer.train()

    preds_output = trainer.predict(test_dataset)
    preds = np.argmax(preds_output.predictions, axis=1)

    f1 = f1_score(y_test, preds, average='macro')
    f1_scores_roberta.append(f1)

print("XLM-R F1 médio:", np.mean(f1_scores_roberta))
print("Desvio padrão:", np.std(f1_scores_roberta))

Map:   0%|          | 0/475 [00:00<?, ? examples/s]

Map:   0%|          | 0/119 [00:00<?, ? examples/s]

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

Map:   0%|          | 0/476 [00:00<?, ? examples/s]

Map:   0%|          | 0/118 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


{'loss': '1.122', 'grad_norm': '3.873', 'learning_rate': '3.639e-05', 'epoch': '0.8333'}
{'loss': '1.092', 'grad_norm': '6.523', 'learning_rate': '2.25e-05', 'epoch': '1.667'}
{'loss': '1.094', 'grad_norm': '4.965', 'learning_rate': '8.611e-06', 'epoch': '2.5'}
{'train_runtime': '43.13', 'train_samples_per_second': '33.11', 'train_steps_per_second': '4.173', 'train_loss': '1.104', 'epoch': '3'}
XLM-R F1 médio: 0.26710723352207427
Desvio padrão: 0.03144687621499999


# Llama 3.1

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes

In [ ]:
from huggingface_hub import login

login("COLOCA AQUI O TOKEN")

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

llama = "meta-llama/Meta-Llama-3-8B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(llama)
tokenizer.padding_side = "left"
tokenizer.pad_token = tokenizer.eos_token

# Configurando para 4-bits
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

llama_model = AutoModelForCausalLM.from_pretrained(
    llama,
    quantization_config=bnb_config,
    device_map="auto"
)

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [ ]:
def classify_llama(text):
    prompt = f"""
Classifique o sentimento do comentário em relação à lei como:
positivo, negativo ou neutro.

Comentário: {text}

Resposta:
"""

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    outputs = llama_model.generate(
        **inputs,
        max_new_tokens=10
    )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    if "positivo" in response.lower():
        return 2
    elif "negativo" in response.lower():
        return 0
    else:
        return 1

In [ ]:
f1_scores_llama = []

for train_idx, test_idx in splits:

    X_test = X.iloc[test_idx]
    y_test = y.iloc[test_idx]

    preds = []

    for text in X_test:
        pred = classify_llama(text)
        preds.append(pred)

    f1 = f1_score(y_test, preds, average='macro')
    f1_scores_llama.append(f1)

print("\n=== RESULTADOS (Llama) ===")
print("F1 médio:", np.mean(f1_scores_llama))
print("Desvio padrão:", np.std(f1_scores_llama))

[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
[transformers] Both `max_new_tokens` (=10) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
[transformers] Both `max_new_tokens` (=10) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refe


=== RESULTADOS (Llama) ===
F1 médio: 0.048870653522960786
Desvio padrão: 0.0022974407461918214
